# 03 - Proyecto Aplicado: Ensambles vs Modelos Base en un Caso Real

## Curso de Machine Learning: Modelos de Clasificación

En este cuaderno dejaremos los datasets de juguete para enfrentarnos a un **problema real de negocio**. Usaremos un dataset de predicción que presenta ruido, variables de distinto tipo (numéricas y categóricas) y cierto nivel de desbalance, simulando escenarios como el riesgo crediticio o la fuga de clientes.

---

### Contenidos
1. Carga de Datos y Análisis Exploratorio Breve
2. Preprocesamiento Profesional con `Pipeline` y `ColumnTransformer`
3. Entrenamiento Comparativo: Random Forest vs Regresión Logística vs KNN
4. Evaluación Orientada a Negocio: ROC-AUC y Curvas Precision-Recall
5. Análisis de la Matriz de Confusión y Costos de Clasificación Errónea
6. Conclusiones sobre la Elección del Modelo


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, precision_recall_curve, auc
)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)
print("Librerías importadas correctamente.")


## 1. Carga de Datos y Configuración del Problema

Para que este cuaderno sea autoejecutable, simularemos de forma controlada un dataset con características idénticas a un caso real de riesgo crediticio, inyectando ruido, desbalance y variables de distintos tipos.

**Objetivo de Negocio:** Identificar correctamente la Clase 1 (alto riesgo / fraude / fuga) frente a la Clase 0.


In [ ]:
# Generación de un dataset estructurado complejo (Simulando un caso real)
from sklearn.datasets import make_classification

X_sim, y_sim = make_classification(
    n_samples=5000, n_features=15, n_informative=8, n_redundant=3,
    weights=[0.85, 0.15],  # Desbalance de clases (85% vs 15%)
    class_sep=0.8, random_state=42
)

# Convertir a DataFrame para simular variables numéricas y categóricas
cols = [f'Num_{i}' for i in range(10)] + [f'Cat_{i}' for i in range(5)]
df = pd.DataFrame(X_sim, columns=cols)
df['Target'] = y_sim

# Simular categorías discretizando algunas columnas continuas
for c in [f'Cat_{i}' for i in range(5)]:
    df[c] = pd.qcut(df[c], q=4, labels=['A', 'B', 'C', 'D'], duplicates='drop')

print("Dimensiones del dataset:", df.shape)
print("\nDistribución del Target:")
print(df['Target'].value_counts(normalize=True).round(3))


## 2. Preprocesamiento Profesional con Pipelines

En un entorno profesional se aplican las transformaciones dentro de un `ColumnTransformer`. Escalaremos las variables numéricas (vital para Regresión Logística y KNN) y aplicaremos codificación (One-Hot Encoding) a las categóricas.


In [ ]:
# Separación X e y
X = df.drop('Target', axis=1)
y = df['Target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

# Identificación de tipos de columnas
num_cols = [c for c in X.columns if X[c].dtype in ['float64', 'int64']]
cat_cols = [c for c in X.columns if X[c].dtype.name == 'category']

# Transformadores
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ])

print(f"Variables numéricas ({len(num_cols)}): {num_cols}")
print(f"Variables categóricas ({len(cat_cols)}): {cat_cols}")


## 3. Entrenamiento Comparativo

Entrenaremos la Regresión Logística, KNN y Random Forest utilizando el mismo esquema de preprocesamiento.


In [ ]:
# Definición de modelos dentro de pipelines
pipelines = {
    'Regresion Logistica': Pipeline(steps=[('prep', preprocessor), ('clf', LogisticRegression(max_iter=1000, random_state=42))]),
    'KNN': Pipeline(steps=[('prep', preprocessor), ('clf', KNeighborsClassifier(n_neighbors=9))]),
    'Random Forest': Pipeline(steps=[('prep', preprocessor), ('clf', RandomForestClassifier(n_estimators=200, max_features='sqrt', random_state=42, n_jobs=-1))])
}

# Entrenamiento y almacenamiento de predicciones
resultados_proba = {}
resultados_clase = {}

for nombre, modelo in pipelines.items():
    print(f"Entrenando {nombre}...")
    modelo.fit(X_train, y_train)
    resultados_clase[nombre] = modelo.predict(X_test)
    resultados_proba[nombre] = modelo.predict_proba(X_test)[:, 1]

print("Entrenamiento finalizado.")


## 4. Evaluación Orientada a Negocio (ROC y Precision-Recall)

Dado el fuerte desbalance de clases, el 'Accuracy' (Exactitud) no es una métrica confiable. Un modelo que prediga que todos son 'Clase 0' tendría 85% de exactitud, pero no aportaría valor.

Utilizaremos **ROC-AUC** para el desempeño global de ordenamiento, y la **Curva Precision-Recall (PR)** para focalizarnos en nuestra capacidad de detectar la clase minoritaria.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Curva ROC
for nombre, probas in resultados_proba.items():
    fpr, tpr, _ = roc_curve(y_test, probas)
    auc_val = roc_auc_score(y_test, probas)
    axes[0].plot(fpr, tpr, lw=2, label=f'{nombre} (AUC = {auc_val:.3f})')

axes[0].plot([0, 1], [0, 1], color='gray', linestyle='--')
axes[0].set_xlabel('Tasa de Falsos Positivos (FPR)')
axes[0].set_ylabel('Tasa de Verdaderos Positivos (TPR / Recall)')
axes[0].set_title('Curva ROC')
axes[0].legend()

# Curva Precision-Recall
for nombre, probas in resultados_proba.items():
    prec, rec, _ = precision_recall_curve(y_test, probas)
    pr_auc = auc(rec, prec)
    axes[1].plot(rec, prec, lw=2, label=f'{nombre} (PR-AUC = {pr_auc:.3f})')

# Baseline para PR curve (proporción de positivos)
baseline = y_test.mean()
axes[1].plot([0, 1], [baseline, baseline], color='gray', linestyle='--', label=f'Baseline ({baseline:.3f})')
axes[1].set_xlabel('Recall (Tasa de Detección)')
axes[1].set_ylabel('Precision (Certeza en la Alerta)')
axes[1].set_title('Curva Precision-Recall (Foco en Clase Minoritaria)')
axes[1].legend()

plt.tight_layout()
plt.show()


## 5. Matrices de Confusión y Costos de Negocio

Analicemos los tipos de errores que comete cada modelo:
- **Falsos Positivos (FP):** Puede significar incomodar al cliente o iniciar procesos de revisión manual innecesarios.
- **Falsos Negativos (FN):** Suele tener el mayor costo directo (fraude exitoso, crédito fallido).


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (nombre, preds) in zip(axes, resultados_clase.items()):
    cm = confusion_matrix(y_test, preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Clase 0', 'Clase 1'])
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'{nombre}\nFP: {cm[0,1]} | FN: {cm[1,0]}', fontweight='bold')
    ax.grid(False)

plt.tight_layout()
plt.show()

# Reporte detallado para el mejor modelo
print("\n--- Reporte de Clasificación (Random Forest) ---")
print(classification_report(y_test, resultados_clase['Random Forest']))


## 6. Conclusiones del Proyecto

1. **Métricas robustas:** Random Forest demostró superioridad para manejar relaciones complejas y desbalance, reflejado en su mayor PR-AUC.
2. **Ajuste para el negocio:** Si se desea minimizar los Falsos Negativos, la ventaja del modelo es que permite ajustar el umbral de decisión para lograr el equilibrio óptimo de costos según las reglas de negocio.
